# 08 · Dates, lags, and panel-shaped tables

**Module:** 2 · **Time:** 45 minutes · **Prerequisites:** Reshaping wide to long and grouped operations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/08_panels_and_time.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F08_panels_and_time.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** These badges link to the currently private repository's `main` after a future merge.

## Learning goals

- **P**arse calendar dates and repeated year identifiers.
- **B**uild a two-wave table with a MultiIndex.
- **C**ompute within-record lags and growth, checking balance.

In [ ]:
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print("Synthetic teaching extract:", survey.shape)

## Why this matters

Panel data combine an entity identifier with a time variable. Lags should be computed within entities after sorting by time. Our sample has a duplicated person ID; therefore, we use a unique *record* key to demonstrate panel mechanics without asserting that those duplicate records are one true individual.

## Parse dates safely

Dates may arrive in mixed formats. Explicit `format='mixed'` works for this small demonstration; real projects should document the source's date convention to avoid ambiguous day/month values.

In [ ]:
dates = pd.Series(["2024-03-01", "03/02/2024", "2024-03-03"])
parsed_dates = pd.to_datetime(dates, format="mixed")
print(parsed_dates.dt.strftime("%Y-%m-%d").tolist())

## Build a long two-wave table

Create a unique row record key and gather annual wages into one column. The record key is not proof of genuine longitudinal tracking.

In [ ]:
wide = survey.reset_index(names="record_id").copy()
long = wide.melt(
    id_vars=["record_id", "person_id"],
    value_vars=["wage_2023", "wage_2024"],
    var_name="wave", value_name="wage_text"
)
long["year"] = long["wave"].str[-4:].astype(int)
long = long.sort_values(["record_id", "year"]).reset_index(drop=True)
print(long.head(4))

## Convert wages, create lags, and calculate growth

Handle the artificial missing code and extreme value before calculating changes, or the resulting growth rates would be misleading.

In [ ]:
numbers = pd.to_numeric(long["wage_text"], errors="coerce")
long["wage"] = numbers.mask(numbers.eq(-99) | numbers.gt(500_000))
long["lag_wage"] = long.groupby("record_id")["wage"].shift(1)
long["growth_pct"] = (long["wage"] / long["lag_wage"] - 1) * 100
print(long[["record_id", "year", "wage", "growth_pct"]].head(6))

## MultiIndex and balance checks

A MultiIndex combines entity and time labels. A balanced two-wave teaching sample should have exactly two rows per record, but this says nothing about real panel validity.

In [ ]:
panel = long.set_index(["record_id", "year"]).sort_index()
wave_counts = long.groupby("record_id")["year"].nunique()
assert panel.index.is_unique
print("Balanced two-wave teaching table:", wave_counts.eq(2).all())
print(panel[["wage", "lag_wage"]].head(4))

## If you know Stata/R

| Task | pandas | Stata | R |
| --- | --- | --- | --- |
| Parse date | `pd.to_datetime(s)` | `date(s, 'YMD')` | `as.Date(s)` |
| Define panel keys | `set_index(['id','year'])` | `xtset id year` | `plm::pdata.frame(...)` |
| Within-entity lag | `groupby('id')['y'].shift(1)` | `L.y` after `xtset` | `dplyr::lag(y)` after sorting/grouping |

A lag in unbalanced annual data may refer to the previous *observed* row, not necessarily the previous calendar year.

## Try it

Fill each answer cell. In the learner edition, incomplete answers print hints without stopping notebook execution.

### ★ Exercise 1 — Parse annual labels

Create `parsed_year` as integers from the last four characters of `long['wave']`.

In [ ]:
parsed_year = None

In [ ]:
expected = long["wave"].str[-4:].astype(int)
if isinstance(parsed_year, pd.Series) and parsed_year.equals(expected):
    print("✓ Years parsed correctly")
else:
    print("Hint: select the last four characters, then call astype(int).")

### ★★ Exercise 2 — Create a panel index

Create `panel_indexed` from long with an index formed by `record_id` and `year`, sorted.

In [ ]:
panel_indexed = None

In [ ]:
expected = long.set_index(["record_id", "year"]).sort_index()
if isinstance(panel_indexed, pd.DataFrame) and panel_indexed.equals(expected):
    print("✓ MultiIndex panel correct")
else:
    print("Hint: use set_index with both keys and sort_index.")

### ★★ Exercise 3 — Compute the within-record lag

Set `previous_wage` to the lagged cleaned wage within record_id.

In [ ]:
previous_wage = None

In [ ]:
expected = long.groupby("record_id")["wage"].shift(1)
if isinstance(previous_wage, pd.Series) and previous_wage.equals(expected):
    print("✓ Lagged wages correct")
else:
    print("Hint: groupby('record_id') before shift(1).")

### ★★★ Exercise 4 — Measure two-wave completeness

Set `balanced` to a Python bool indicating every record_id appears in both years.

In [ ]:
balanced = None

In [ ]:
expected = bool(long.groupby("record_id")["year"].nunique().eq(2).all())
if isinstance(balanced, bool) and balanced == expected:
    print("✓ Panel balance check correct")
else:
    print("Hint: count distinct years within each record and check they equal two.")

## Common errors

- Sorting only by year mixes different entities and creates incorrect lag values.
- An unverified respondent key is not a valid panel identifier, even if the software accepts it.
- Mixed date strings can be ambiguous; document day/month ordering and check parsed dates.
- A previous observed value from `.shift()` need not be the previous calendar period when years are missing.

## Recap

- Panel-shaped data use entity and time keys together.
- Parse and validate time variables before lags and growth calculations.
- Sort by entity and time, then compute lags within entity.
- Check balance and identifier validity before making econometric claims.

## Next steps

Next: [Working with larger data](09_larger_data.ipynb). See the [pandas time-series guide](https://pandas.pydata.org/docs/user_guide/timeseries.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md).